# Train + MLflow tracking

- Logs metrics, signatures, and a sklearn model artifact under MLflow experiment `/Shared/portfolio_renewal_demo` on Databricks (short name locally).
- On Databricks, point `MLFLOW_TRACKING_URI` at the Workspace registry (normally automatic).
- Optional **Model Registry**: set env **`REGISTER_MODEL_NAME`** (e.g. `catalog.schema.renewal_classifier` or workspace model name) so `train_and_register` passes `registered_model_name` into `mlflow.sklearn.log_model` (requires registry permissions).
- For constrained sandboxes without MLflow, run `scripts/run_local_pipeline.py` (pickle fallback) locally.

In [ ]:
%pip install -q pandas pyarrow scikit-learn mlflow joblib numpy
import os, sys
REPO = os.environ.get("PORTFOLIO_REPO_ROOT", "../..")
sys.path.insert(0, f"{REPO}/python")
if os.environ.get("PORTFOLIO_BASE_PATH") is None and "dbutils" in globals():
    dbutils.widgets.text("base_path", "")  # type: ignore
    _bp = dbutils.widgets.get("base_path")  # type: ignore
    if _bp:
        os.environ["PORTFOLIO_BASE_PATH"] = _bp

from cs_portfolio.train import train_and_register
print(train_and_register())

## Unity Catalog / Model Registry

`train_and_register` already passes **`registered_model_name`** to `mlflow.sklearn.log_model` when you set env **`REGISTER_MODEL_NAME`** (or `MLFLOW_REGISTER_MODEL_NAME`) — e.g. `main.demo.renewal_propensity_classifier` on a Databricks workspace with registry permissions. Leave unset for file-store-only runs.

See **`databricks/docs/PRODUCTIONIZATION_UC_MLFLOW.md`** for aliases (champion/challenger), batch scoring from `models:/…`, rollback, and monitoring.